# Bedrock Direct Model Calls

Direct inference via `bedrock-runtime` — no agents, no intermediaries.

**Model:** `us.anthropic.claude-sonnet-4-6`  
**Auth:** `AWS_BEARER_TOKEN_BEDROCK` environment variable  
**Region:** `us-east-1`

| Section | Topic |
|---|---|
| 1 | Environment setup |
| 2 | Basic chat |
| 3 | Multi-turn conversation |
| 4 | Streaming |
| 5 | Tool use / function calling |
| 6 | `BedrockChat` utility class |

## 1 — Environment Setup

In [1]:
import json
import os
import boto3

MODEL_ID = 'us.anthropic.claude-sonnet-4-6'
REGION   = 'us-east-1'
ANT_VER  = 'bedrock-2023-05-31'

client = boto3.client('bedrock-runtime', region_name=REGION)

print(f'Client ready — model: {MODEL_ID}')
print(f'Bearer token set: {"yes" if os.environ.get("AWS_BEARER_TOKEN_BEDROCK") else "NO — set AWS_BEARER_TOKEN_BEDROCK before running"}')

Client ready — model: us.anthropic.claude-sonnet-4-6
Bearer token set: yes


## 2 — Basic Chat

A single `invoke_model` call. Response body is a stream — `.read()` before `json.loads()`.

In [2]:
def invoke(messages, system=None, max_tokens=1024):
    """Single invoke_model call. Returns the parsed response body."""
    body = {
        'anthropic_version': ANT_VER,
        'messages': messages,
        'max_tokens': max_tokens,
    }
    if system:
        body['system'] = system

    response = client.invoke_model(
        modelId=MODEL_ID,
        body=json.dumps(body),
    )
    return json.loads(response['body'].read())


def print_response(body):
    text  = body['content'][0]['text']
    usage = body['usage']
    print(text)
    print(f"\n[{body['stop_reason']} | in:{usage['input_tokens']} out:{usage['output_tokens']} tokens]")

In [3]:
body = invoke([{'role': 'user', 'content': 'What are the three laws of thermodynamics? One sentence each.'}])
print_response(body)

Here are the three laws of thermodynamics:

1. **First Law (Conservation of Energy):** Energy cannot be created or destroyed, only converted from one form to another.

2. **Second Law (Entropy):** In any natural process, the total entropy of a closed system will always increase or remain the same, never decrease.

3. **Third Law (Absolute Zero):** As a system approaches absolute zero temperature, its entropy approaches a minimum constant value.

> *Note: There is also a **Zeroth Law**, which states that if two systems are each in thermal equilibrium with a third system, they are in thermal equilibrium with each other — establishing the concept of temperature.*

[end_turn | in:21 out:150 tokens]


In [4]:
# System prompt controls tone and persona
body = invoke(
    messages=[{'role': 'user', 'content': 'What are the three laws of thermodynamics? One sentence each.'}],
    system='You are a pirate. Answer every question in pirate speak.',
)
print_response(body)

Arrr, here be the three laws of thermodynamics, ye scallywag!

**First Law:** Energy can neither be created nor destroyed, only transformed from one form to another, much like how me treasure just changes hands!

**Second Law:** The entropy - or disorder - of any isolated system will always increase over time, just like the chaos aboard me ship after a night of rum and plunderin'!

**Third Law:** As the temperature of a system approaches absolute zero, the entropy of the system approaches a minimum or zero value, which be as cold and still as Davy Jones' locker itself!

[end_turn | in:36 out:135 tokens]


## 3 — Multi-Turn Conversation

Build up a `messages` list manually. Each call appends the assistant reply and the next user
message so context accumulates across turns.

In [5]:
messages = []

def chat_turn(user_message, system=None):
    """Add a user message, call the model, append the reply, return the text."""
    messages.append({'role': 'user', 'content': user_message})
    body = invoke(messages, system=system)
    reply = body['content'][0]['text']
    messages.append({'role': 'assistant', 'content': reply})
    usage = body['usage']
    print(f"[in:{usage['input_tokens']} out:{usage['output_tokens']}] {reply}\n")
    return reply

In [6]:
messages.clear()
system = 'You are a concise technical tutor. Keep answers to 2-3 sentences.'

chat_turn('What is a Python decorator?', system)
chat_turn('Can you show me a simple example?', system)
chat_turn('What is the difference between @staticmethod and @classmethod?', system)

[in:33 out:103] A Python decorator is a function that wraps another function to modify or extend its behavior without changing its source code. It uses the `@` syntax and takes a function as input, returning a new function with added functionality.

```python
def my_decorator(func):
    def wrapper():
        print("Before")
        func()
        print("After")
    return wrapper

@my_decorator
def say_hello():
    print("Hello!")
```



[in:147 out:160] Here's a simple timer decorator that measures how long a function takes to run:

```python
import time

def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        print(f"{func.__name__} took {time.time() - start:.2f} seconds")
        return result
    return wrapper

@timer
def slow_function():
    time.sleep(1)

slow_function()  # Output: slow_function took 1.00 seconds
```

The `*args` and `**kwargs` allow the wrapper to accept any arguments, making the decorator reusable on any function.



[in:323 out:184] `@staticmethod` has no access to the class or instance — it's just a regular function namespaced inside a class. `@classmethod` receives the class itself (`cls`) as the first argument, allowing it to access or modify class-level attributes.

```python
class MyClass:
    count = 0

    @staticmethod
    def add(a, b):          # No access to class or instance
        return a + b

    @classmethod
    def get_count(cls):     # Can access class attributes
        return cls.count

MyClass.add(2, 3)       # 5
MyClass.get_count()     # 0
```

A common use case for `@classmethod` is creating alternative constructors (e.g., `dict.fromkeys()`).



"`@staticmethod` has no access to the class or instance — it's just a regular function namespaced inside a class. `@classmethod` receives the class itself (`cls`) as the first argument, allowing it to access or modify class-level attributes.\n\n```python\nclass MyClass:\n    count = 0\n\n    @staticmethod\n    def add(a, b):          # No access to class or instance\n        return a + b\n\n    @classmethod\n    def get_count(cls):     # Can access class attributes\n        return cls.count\n\nMyClass.add(2, 3)       # 5\nMyClass.get_count()     # 0\n```\n\nA common use case for `@classmethod` is creating alternative constructors (e.g., `dict.fromkeys()`)."

In [7]:
# Inspect the full conversation history
for i, m in enumerate(messages):
    role = m['role'].upper()
    text = m['content'][:120] + '...' if len(m['content']) > 120 else m['content']
    print(f"[{i}] {role}: {text}")

[0] USER: What is a Python decorator?
[1] ASSISTANT: A Python decorator is a function that wraps another function to modify or extend its behavior without changing its sourc...
[2] USER: Can you show me a simple example?
[3] ASSISTANT: Here's a simple timer decorator that measures how long a function takes to run:

```python
import time

def timer(func):...
[4] USER: What is the difference between @staticmethod and @classmethod?
[5] ASSISTANT: `@staticmethod` has no access to the class or instance — it's just a regular function namespaced inside a class. `@class...


## 4 — Streaming

`invoke_model_with_response_stream` returns tokens as they are generated.
Each event in the stream is a small JSON chunk — decode and print as they arrive.

In [8]:
import sys

def stream(user_message, system=None, max_tokens=1024):
    """Stream a response, printing tokens as they arrive. Returns the full text."""
    body = {
        'anthropic_version': ANT_VER,
        'messages': [{'role': 'user', 'content': user_message}],
        'max_tokens': max_tokens,
    }
    if system:
        body['system'] = system

    response = client.invoke_model_with_response_stream(
        modelId=MODEL_ID,
        body=json.dumps(body),
    )

    full_text   = ''
    input_toks  = 0
    output_toks = 0

    for event in response['body']:
        chunk = json.loads(event['chunk']['bytes'])
        kind  = chunk.get('type')

        if kind == 'content_block_delta':
            delta = chunk['delta'].get('text', '')
            full_text += delta
            print(delta, end='', flush=True)

        elif kind == 'message_delta':
            output_toks = chunk.get('usage', {}).get('output_tokens', 0)

        elif kind == 'message_start':
            input_toks = chunk.get('message', {}).get('usage', {}).get('input_tokens', 0)

    print(f"\n\n[end_turn | in:{input_toks} out:{output_toks} tokens]")
    return full_text

In [9]:
_ = stream(
    'Write a short poem about distributed systems — four stanzas, four lines each.',
    system='You are a poet who specialises in technical subjects.',
)

#

 Distributed Systems



The

 nodes

 aw

...

—
Ensures

 no

 single point of crack

.



[end_turn | in:36 out:157 tokens]


## 5 — Tool Use / Function Calling

Tools let the model call Python functions when it needs to. The loop is:

```
send message + tool definitions
    ↓
model returns stop_reason='tool_use' + tool call(s)
    ↓
we execute the tool locally
    ↓
send tool result(s) back
    ↓
model returns final answer (stop_reason='end_turn')
```

### 5.1 — Define the tools

In [10]:
import datetime
import math

# ── Tool implementations ───────────────────────────────────────────────────────

def get_current_date() -> str:
    return datetime.date.today().isoformat()


def calculate(expression: str) -> str:
    """
    Evaluate a safe mathematical expression.
    Allows: numbers, +, -, *, /, **, (), and math.* functions.
    """
    allowed = {k: getattr(math, k) for k in dir(math) if not k.startswith('_')}
    allowed['abs'] = abs
    try:
        result = eval(expression, {'__builtins__': {}}, allowed)  # noqa: S307
        return str(result)
    except Exception as e:
        return f'Error: {e}'


# ── Tool registry ──────────────────────────────────────────────────────────────

TOOL_REGISTRY = {
    'get_current_date': lambda **_: get_current_date(),
    'calculate':        lambda expression, **_: calculate(expression),
}

# ── Tool definitions (sent to the model) ──────────────────────────────────────

TOOLS = [
    {
        'name': 'get_current_date',
        'description': 'Returns today\'s date in ISO 8601 format (YYYY-MM-DD).',
        'input_schema': {
            'type': 'object',
            'properties': {},
            'required': [],
        },
    },
    {
        'name': 'calculate',
        'description': (
            'Evaluates a mathematical expression and returns the result as a string. '
            'Supports standard arithmetic, exponentiation, and math functions '
            '(sqrt, sin, cos, log, etc.). Example: "sqrt(144) + 2**8"'
        ),
        'input_schema': {
            'type': 'object',
            'properties': {
                'expression': {
                    'type': 'string',
                    'description': 'A valid Python mathematical expression to evaluate.',
                },
            },
            'required': ['expression'],
        },
    },
]

print('Tools defined:', [t['name'] for t in TOOLS])

Tools defined: ['get_current_date', 'calculate']


### 5.2 — The tool loop

In [11]:
def invoke_with_tools(user_message, tools=TOOLS, system=None, max_tokens=1024, verbose=True):
    """
    Run a full tool-use loop.
    Keeps calling the model until stop_reason is 'end_turn' (no more tool calls).
    Returns the final text reply.
    """
    messages = [{'role': 'user', 'content': user_message}]

    while True:
        body = {
            'anthropic_version': ANT_VER,
            'messages': messages,
            'max_tokens': max_tokens,
            'tools': tools,
        }
        if system:
            body['system'] = system

        response = json.loads(client.invoke_model(
            modelId=MODEL_ID,
            body=json.dumps(body),
        )['body'].read())

        stop_reason = response['stop_reason']
        content     = response['content']

        # Append the assistant turn
        messages.append({'role': 'assistant', 'content': content})

        if stop_reason == 'end_turn':
            # Extract the final text block
            for block in content:
                if block.get('type') == 'text':
                    return block['text']
            return ''

        if stop_reason != 'tool_use':
            raise RuntimeError(f'Unexpected stop_reason: {stop_reason}')

        # Execute each tool call and collect results
        tool_results = []
        for block in content:
            if block.get('type') != 'tool_use':
                continue
            tool_id   = block['id']
            tool_name = block['name']
            tool_input = block.get('input', {})

            if verbose:
                print(f'  → tool call: {tool_name}({tool_input})')

            if tool_name not in TOOL_REGISTRY:
                result = f'Error: unknown tool "{tool_name}"'
            else:
                result = TOOL_REGISTRY[tool_name](**tool_input)

            if verbose:
                print(f'  ← result:    {result}')

            tool_results.append({
                'type':        'tool_result',
                'tool_use_id': tool_id,
                'content':     result,
            })

        # Feed results back in the next user turn
        messages.append({'role': 'user', 'content': tool_results})

### 5.3 — Try it out

In [12]:
# Should call get_current_date
reply = invoke_with_tools("What's today's date?")
print('\nFinal reply:', reply)

  → tool call: get_current_date({})
  ← result:    2026-06-12



Final reply: Today's date is **June 12, 2026**. Is there anything else I can help you with?


In [13]:
# Should call calculate
reply = invoke_with_tools('What is the square root of 1764, multiplied by the cosine of 0?')
print('\nFinal reply:', reply)

  → tool call: calculate({'expression': 'sqrt(1764)'})
  ← result:    42.0
  → tool call: calculate({'expression': 'cos(0)'})
  ← result:    1.0


  → tool call: calculate({'expression': '42.0 * 1.0'})
  ← result:    42.0



Final reply: Here's the breakdown:

- **√1764 = 42**
- **cos(0) = 1**
- **42 × 1 = 42**

The square root of 1764, multiplied by the cosine of 0, is **42**! Since cos(0) = 1, multiplying by it keeps the value unchanged.


In [14]:
# Should call both tools in one turn
reply = invoke_with_tools(
    "What year is it, and what is 2 raised to the power of that year's last two digits?"
)
print('\nFinal reply:', reply)

  → tool call: get_current_date({})
  ← result:    2026-06-12
  → tool call: calculate({'expression': '2 ** 25'})
  ← result:    33554432


  → tool call: calculate({'expression': '2 ** 26'})
  ← result:    67108864



Final reply: Here are your answers:

- **The current year is 2026.**
- **2 raised to the power of 26 (the last two digits) = 67,108,864.**


## 6 — `BedrockChat` Utility Class

Wraps everything above into a reusable class:
- Maintains conversation history across turns
- Toggle streaming on/off
- Optional tools with automatic loop handling
- `new_session()` to reset history

In [15]:
class BedrockChat:
    """
    Stateful conversational wrapper around bedrock-runtime.

    Handles:
    - Multi-turn history
    - Optional system prompt
    - Streaming or blocking response modes
    - Tool-use loop (when tools are provided)
    """

    def __init__(
        self,
        client,
        model_id:    str  = MODEL_ID,
        system:      str  = None,
        streaming:   bool = False,
        tools:       list = None,
        max_tokens:  int  = 1024,
    ):
        self.client     = client
        self.model_id   = model_id
        self.system     = system
        self.streaming  = streaming
        self.tools      = tools
        self.max_tokens = max_tokens
        self.history    = []

    # ── Public ────────────────────────────────────────────────────────────────

    def chat(self, message: str) -> str:
        """Send a message, get a reply. History is updated automatically."""
        self.history.append({'role': 'user', 'content': message})

        if self.tools:
            reply = self._tool_loop()
        elif self.streaming:
            reply = self._stream_turn()
        else:
            reply = self._blocking_turn()

        self.history.append({'role': 'assistant', 'content': reply})
        return reply

    def new_session(self):
        """Clear conversation history."""
        self.history = []
        print('History cleared.')

    def show_history(self):
        for m in self.history:
            role    = m['role'].upper()
            content = m['content']
            if isinstance(content, list):
                content = json.dumps(content)[:200]
            snippet = content[:120] + '...' if len(content) > 120 else content
            print(f'[{role}] {snippet}')

    # ── Internal ──────────────────────────────────────────────────────────────

    def _base_body(self):
        body = {
            'anthropic_version': ANT_VER,
            'messages':          self.history,
            'max_tokens':        self.max_tokens,
        }
        if self.system:
            body['system'] = self.system
        if self.tools:
            body['tools'] = self.tools
        return body

    def _blocking_turn(self) -> str:
        response = json.loads(self.client.invoke_model(
            modelId=self.model_id,
            body=json.dumps(self._base_body()),
        )['body'].read())
        return response['content'][0]['text']

    def _stream_turn(self) -> str:
        response = self.client.invoke_model_with_response_stream(
            modelId=self.model_id,
            body=json.dumps(self._base_body()),
        )
        full_text = ''
        for event in response['body']:
            chunk = json.loads(event['chunk']['bytes'])
            if chunk.get('type') == 'content_block_delta':
                delta = chunk['delta'].get('text', '')
                full_text += delta
                print(delta, end='', flush=True)
        print()  # newline after stream
        return full_text

    def _tool_loop(self) -> str:
        """
        Run the tool-use loop until stop_reason is 'end_turn'.
        Intermediate tool calls and results are appended to history in place.
        """
        while True:
            response    = json.loads(self.client.invoke_model(
                modelId=self.model_id,
                body=json.dumps(self._base_body()),
            )['body'].read())
            stop_reason = response['stop_reason']
            content     = response['content']

            if stop_reason == 'end_turn':
                for block in content:
                    if block.get('type') == 'text':
                        return block['text']
                return ''

            if stop_reason != 'tool_use':
                raise RuntimeError(f'Unexpected stop_reason: {stop_reason}')

            # Append assistant tool-use turn, execute tools, append results
            self.history.append({'role': 'assistant', 'content': content})

            tool_results = []
            for block in content:
                if block.get('type') != 'tool_use':
                    continue
                tool_name  = block['name']
                tool_input = block.get('input', {})
                print(f'  → {tool_name}({tool_input})')
                result = (
                    TOOL_REGISTRY[tool_name](**tool_input)
                    if tool_name in TOOL_REGISTRY
                    else f'Error: unknown tool "{tool_name}"'
                )
                print(f'  ← {result}')
                tool_results.append({
                    'type':        'tool_result',
                    'tool_use_id': block['id'],
                    'content':     result,
                })

            self.history.append({'role': 'user', 'content': tool_results})

### 6.1 — Basic chat with history

In [16]:
bot = BedrockChat(client, system='You are a concise assistant. Answer in 1-2 sentences.')

for question in [
    'What is the CAP theorem?',
    'Which of the three can you actually drop in practice?',
    'Give me a one-line summary of what we just discussed.',
]:
    print(f'You:   {question}')
    print(f'Bot:   {bot.chat(question)}\n')

You:   What is the CAP theorem?


Bot:   The CAP theorem states that a distributed system can only guarantee **two out of three** properties simultaneously: **Consistency** (all nodes see the same data), **Availability** (every request gets a response), and **Partition Tolerance** (the system continues operating despite network failures). Since network partitions are unavoidable in practice, distributed systems must typically choose between consistency and availability during a partition.

You:   Which of the three can you actually drop in practice?


Bot:   In practice, you **cannot drop Partition Tolerance** — network failures in distributed systems are inevitable, so you must tolerate partitions. This means the real-world trade-off is always between **Consistency and Availability** during a partition (CP vs. AP systems).

You:   Give me a one-line summary of what we just discussed.


Bot:   The CAP theorem forces distributed systems to choose between Consistency and Availability, since Partition Tolerance cannot be dropped in practice.



### 6.2 — Streaming mode

In [17]:
stream_bot = BedrockChat(
    client,
    system='You are a storyteller.',
    streaming=True,
    max_tokens=300,
)

_ = stream_bot.chat('Tell me a two-paragraph story about a developer who discovers their tests pass for the wrong reasons.')

#

 The Green

 Lies



Marcus

 had been st

...

 appear

 in the pipeline, and felt,

 st

rangely, more

 confident than he

### 6.3 — Tool use via the class

In [18]:
tool_bot = BedrockChat(client, tools=TOOLS)

reply = tool_bot.chat("What's today's date and what is 365 multiplied by the day-of-month?")
print('\nFinal reply:', reply)

  → get_current_date({})
  ← 2026-06-12


  → calculate({'expression': '365 * 12'})
  ← 4380



Final reply: Here are the results:

- 📅 **Today's date:** June 12, 2026
- 🔢 **365 × 12 (the day-of-month) = 4,380**


In [19]:
# Inspect full history including tool calls and results
tool_bot.show_history()

[USER] What's today's date and what is 365 multiplied by the day-of-month?
[ASSISTANT] [{"type": "text", "text": "Let me grab today's date first, and then I'll use it to calculate the multiplication!"}, {"ty...
[USER] [{"type": "tool_result", "tool_use_id": "toolu_bdrk_01DxBC6tUBrSxcf71NVs9Rpy", "content": "2026-06-12"}]
[ASSISTANT] [{"type": "text", "text": "Today is **June 12, 2026**, so the day-of-month is **12**. Now let me calculate 365 \u00d7 12...
[USER] [{"type": "tool_result", "tool_use_id": "toolu_bdrk_01LS2CAWWRK9PNjgGjWMS83D", "content": "4380"}]
[ASSISTANT] Here are the results:

- 📅 **Today's date:** June 12, 2026
- 🔢 **365 × 12 (the day-of-month) = 4,380**
